In [4]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()
openai_client = OpenAI()

In [5]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [6]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [7]:
assistant.rag("How do I run Olama?")

'The provided context does not include specific instructions on how to run Olama. Please refer to the appropriate documentation or resources related to Olama for guidance on running it. If you have any other questions or need assistance with a different topic, feel free to ask!'

In [8]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}


In [9]:
messages = [
    {"role": "user", "content": "How do I use Olama"}
]
response = openai_client.responses.create(
    model="gpt-4o-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"Olama"}', call_id='call_VzaZEgaYT5e8mVCLenE2TdHc', name='search', type='function_call', id='fc_0f1cb05bf5e773de006a26f6afe9d4819a827e5010b961870e', namespace=None, status='completed')]

In [10]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [11]:
result_json

'[]'

In [12]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [13]:
messages

[{'role': 'user', 'content': 'How do I use Olama'},
 ResponseFunctionToolCall(arguments='{"query":"Olama"}', call_id='call_VzaZEgaYT5e8mVCLenE2TdHc', name='search', type='function_call', id='fc_0f1cb05bf5e773de006a26f6afe9d4819a827e5010b961870e', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_VzaZEgaYT5e8mVCLenE2TdHc',
  'output': '[]'}]

In [14]:
response = openai_client.responses.create(
    model="gpt-4o-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'It seems there\'s no specific information available on "Olama" in the FAQ database. However, I can provide general tips on how to use software or platforms if you give me more context. \n\nIs "Olama" a specific app, platform, or tool? Let me know so I can assist you better!'

### Agentic Loop

In [15]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [16]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [17]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-4o-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join the course"}


In [18]:
messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join the course"}', call_id='call_MUeVmQ24xyyJ7Z8QnSqyoayk', name='search', type='function_call', id='fc_0952c5aaa095f14a006a26f6c7569c819980f2794435047879', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_MUeVmQ24xyyJ7Z8QnSqyoayk',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "Ge

In [19]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [20]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Ollama local run install start locally course FAQ"}
function_call: search {"query":"Olama locally run install start model localhost FAQ"}
iteration #2...
ASSISTANT:
To run Ollama locally:

1. **Install Ollama**
   - **macOS**: download and install from https://ollama.com/download
   - **Windows**: download and install the `.msi` from the same page
   - **Linux**:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model locally**
   ```bash
   ollama run llama3
   ```
   This will download the model if needed and open a local chat interface.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should see a response showing Ollama is available.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": "Hello!"}

'To run Ollama locally:\n\n1. **Install Ollama**\n   - **macOS**: download and install from https://ollama.com/download\n   - **Windows**: download and install the `.msi` from the same page\n   - **Linux**:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a model locally**\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model if needed and open a local chat interface.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should see a response showing Ollama is available.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you want, I can also show you how to run a different model, use Ollama with Docker, or call it from a noteb

### Guardrail

In [21]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit queen's gambit course FAQ"}
iteration #2...
function_call: search {"query":"queen gambit chess queen's gambit FAQ"}
iteration #3...
ASSISTANT:
I couldn’t find any course FAQ entry about “queen’s gambit,” and it looks like this may be off-topic for the course.

If you meant something course-related, feel free to rephrase it and I’ll check the FAQ again. Is there another area you want to explore?


'I couldn’t find any course FAQ entry about “queen’s gambit,” and it looks like this may be off-topic for the course.\n\nIf you meant something course-related, feel free to rephrase it and I’ll check the FAQ again. Is there another area you want to explore?'